In [ ]:
"""
노인복지시설 데이터의 각 연도 4분기 수치를 연도 대표값으로 반환
"""

import pandas as pd
from pathlib import Path

# 1. 파일 경로 설정 (사용자 경로에 맞춤)
RAW_DIR = Path(r"C:\rookies6\medical-infrastructure-viz\data\aging\raw")
INPUT_FILE = RAW_DIR / "kosis_medical_institutions_raw.csv"
OUTPUT_FILE = RAW_DIR / "kosis_medical_institutions_yearly.csv"

def aggregate_quarterly_to_yearly(mode="q4"):
    """
    분기별 요양기관 현황 데이터를 연도별 데이터로 합치는 함수
    
    Parameters:
      mode (str): 
        - 'q4'   : 각 연도의 4분기(기말) 수치를 연도 대표값으로 사용 (권장)
        - 'mean' : 4개 분기 수치의 연평균을 산출하여 정수로 반환
    """
    print(f"📂 파일 읽기 시작: {INPUT_FILE}")
    
    # KOSIS 원본 CSV 불러오기 (한글 인코딩 utf-8 또는 cp949 지원)
    try:
        df_raw = pd.read_csv(INPUT_FILE, encoding="utf-8", header=None)
    except UnicodeDecodeError:
        df_raw = pd.read_csv(INPUT_FILE, encoding="cp949", header=None)

    # 헤더 정보 추출
    periods = df_raw.iloc[0].values       # 행 0: 2015.1/4, 2015.2/4 ...
    cat_large = df_raw.iloc[1].values     # 행 1: 계, 의료기관, 보건기관, 약국
    cat_sub = df_raw.iloc[2].values       # 행 2: 소계, 상급종합병원, 종합병원, 의원...

    # 데이터 영역 (행 3부터 지역 데이터: 계, 서울, 부산...)
    data_rows = df_raw.iloc[3:].copy()
    regions = data_rows.iloc[:, 0].values # 시도별 이름 (서울, 부산 등)

    # 연도 목록 추출 (2015 ~ 2024)
    years = sorted(list(set([p.split('.')[0] for p in periods if '.' in str(p)])))
    
    # 신규 연도별 데이터프레임 구성을 위한 헤더 생성
    # 첫 3개 행(설명 헤더) 복사
    new_cols_info = [('시도별(1)', '시도별(1)', '시도별(1)')]
    
    # 기준 분기 결정 (Mode에 따른 분기 컬럼 인덱스 추출)
    yearly_data_dict = {'시도별(1)': regions}

    # 17개 시설 구분 항목 추출 (첫 분기 블록 참조)
    sub_categories = cat_sub[1:17] # 소계, 상급종합병원, 종합병원, 병원, 요양병원, 의원 ...

    for year in years:
        for cat_l, cat_s in zip(cat_large[1:17], cat_sub[1:17]):
            col_name = f"{year}_{cat_l}_{cat_s}"
            
            if mode == "q4":
                # 해당 연도의 4분기 컬럼 찾기 (예: '2015.4/4')
                target_q = f"{year}.4/4"
                matching_cols = [
                    idx for idx, (p, s) in enumerate(zip(periods, cat_sub)) 
                    if p == target_q and s == cat_s
                ]
                if matching_cols:
                    col_idx = matching_cols[0]
                    yearly_data_dict[col_name] = pd.to_numeric(data_rows.iloc[:, col_idx], errors='coerce').fillna(0).astype(int).values
            
            elif mode == "mean":
                # 1~4분기 4개 컬럼의 평균값 계산
                target_prefix = f"{year}."
                matching_cols = [
                    idx for idx, (p, s) in enumerate(zip(periods, cat_sub)) 
                    if str(p).startswith(target_prefix) and s == cat_s
                ]
                if matching_cols:
                    sub_df = data_rows.iloc[:, matching_cols].apply(pd.to_numeric, errors='coerce')
                    yearly_data_dict[col_name] = sub_df.mean(axis=1).round().astype(int).values

    # 새로운 연도별 DataFrame 생성
    df_yearly = pd.DataFrame(yearly_data_dict)

    # CSV 파일 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용)
    df_yearly.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"✅ 연도별 변환 완료! 파일 저장 위치: {OUTPUT_FILE}")
    print(f"📊 변환 후 데이터 크기: {df_yearly.shape} (행: {len(df_yearly)}, 열: {len(df_yearly.columns)})")
    
    return df_yearly

if __name__ == "__main__":
    # 4분기(기말) 기준으로 합치기 실행
    df_result = aggregate_quarterly_to_yearly(mode="q4")
    
    # 미리보기 출력
    print("\n[변환 결과 데이터 미리보기]")
    print(df_result.iloc[:5, :6])

📂 파일 읽기 시작: C:\rookies6\medical-infrastructure-viz\data\aging\raw\kosis_medical_institutions_raw.csv
✅ 연도별 변환 완료! 파일 저장 위치: C:\rookies6\medical-infrastructure-viz\data\aging\raw\kosis_medical_institutions_yearly.csv
📊 변환 후 데이터 크기: (18, 161) (행: 18, 열: 161)

[변환 결과 데이터 미리보기]
  시도별(1)  2015_계_소계  2015_의료기관_상급종합병원  2015_의료기관_종합병원  2015_의료기관_병원  \
0      계      88163                43             294          1496   
1  서울특별시      21507                14              42           218   
2  부산광역시       6416                 4              24           130   
3  대구광역시       4706                 4               8           114   
4  인천광역시       4123                 3              16            55   

   2015_의료기관_요양병원  
0            1372  
1             103  
2             190  
3              61  
4              64  


In [ ]:
"""
노인복지시설 연도별 데이터 중 [요양병원] 컬럼만 추출
"""
from pathlib import Path
import pandas as pd

# 1. 파일 경로 설정
RAW_DIR = Path(r"C:\rookies6\medical-infrastructure-viz\data\aging\raw")
INPUT_FILE = RAW_DIR / "kosis_medical_institutions_yearly.csv"
OUTPUT_FILE = RAW_DIR / "kosis_nursing_hospitals_yearly.csv"

# 2. 파일 불러오기
try:
    df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
except FileNotFoundError:
    df = pd.read_csv(INPUT_FILE, encoding="utf-8")

# 3. '시도별(1)' 컬럼과 '요양병원'이 포함된 컬럼만 필터링
target_cols = [col for col in df.columns if "시도별" in col or "요양병원" in col]
df_nursing_hospital = df[target_cols].copy()

# 4. (선택사항) 컬럼명을 보기 쉽게 정제 (예: '2015_의료기관_요양병원' -> '2015년')
# df_nursing_hospital.columns = [col.split('_')[0] + '년' if '_' in col else col for col in df_nursing_hospital.columns]

# 5. 새로운 CSV 파일로 저장
df_nursing_hospital.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"✅ 추출 완료! 파일 저장 위치: {OUTPUT_FILE}")
print("\n[추출된 요양병원 데이터 미리보기]")
print(df_nursing_hospital.head())

✅ 추출 완료! 파일 저장 위치: C:\rookies6\medical-infrastructure-viz\data\aging\raw\kosis_nursing_hospitals_yearly.csv

[추출된 요양병원 데이터 미리보기]
  시도별(1)  2015_의료기관_요양병원  2016_의료기관_요양병원  2017_의료기관_요양병원  2018_의료기관_요양병원  \
0      계            1372            1428            1529            1560   
1  서울특별시             103             110             115             119   
2  부산광역시             190             197             197             187   
3  대구광역시              61              62              62              64   
4  인천광역시              64              68              72              72   

   2019_의료기관_요양병원  2020_의료기관_요양병원  2021_의료기관_요양병원  2022_의료기관_요양병원  \
0            1577            1582            1464            1435   
1             124             127             124             122   
2             190             187             169             163   
3              69              71              74              76   
4              71              70              67              66   
